# ICD ReAct Tool Playground

Use this notebook to call the ICD manual-browser tools directly. The same config resolution will be reused later by the ReAct baseline, so this is the quickest way to inspect the tool payloads before wiring the agent loop.

When EXECUTION_ENV=databricks, the notebook defaults to the catalog-backed manual root at /Volumes/usdo_aa_catalog/research_tam_datasets/mimic/ICD_manual.

In [ ]:
from __future__ import annotations

import importlib
import json
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not find repo root containing pyproject.toml.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import baselines.icd_react as icd_react
import baselines.icd_react.tools as icd_react_tools

importlib.reload(icd_react_tools)
importlib.reload(icd_react)

from baselines.icd_react import (
    list_guideline_toc,
    list_index_main_terms,
    list_tabular_chapters,
    load_config,
    open_guideline_section,
    open_index_term,
    open_tabular_chapter,
    open_tabular_entry,
    open_tabular_section,
)

In [ ]:
config = load_config()
summary = {
    "execution_env": config.execution_env,
    "manuals_root": str(config.manuals_root),
    "guidelines_pdf_path": str(config.guidelines_pdf_path),
    "index_xml_path": str(config.index_xml_path),
    "tabular_xml_path": str(config.tabular_xml_path),
}
print(json.dumps(summary, indent=2))

## 1. Browse Index Main Terms

This is the high-level entry point. Use a letter and or prefix instead of dumping the whole Index.

In [ ]:
index_matches = list_index_main_terms(config=config, letter="A", prefix="", limit=1000)
print(json.dumps(index_matches, indent=2))

## 2. Open One Index Term

Pick an entry_id from the previous cell and inspect the subtree.

In [ ]:
entry_id = index_matches["results"][3]["entry_id"]
index_entry = open_index_term(config=config, entry_id=entry_id)
print(json.dumps(index_entry, indent=2)[:12000])

## 3. Open a Tabular Entry

Use any exact ICD code to inspect its hierarchy and local notes.

In [ ]:
tabular_code = "A00"
tabular_entry = open_tabular_entry(config=config, code=tabular_code)
print(json.dumps(tabular_entry, indent=2)[:12000])

## 4. Browse Guideline TOC

The current TOC is extracted heuristically from the FY2019 PDF, so treat it as a v0 browser rather than a final benchmark representation.

In [ ]:
guideline_toc = list_guideline_toc(config=config, limit=10)
print(json.dumps(guideline_toc, indent=2)[:12000])

## 5. Open a Guideline Section

Take a section_id from the TOC results and inspect the extracted section text.

In [ ]:
section_id = guideline_toc["results"][0]["section_id"]
guideline_section = open_guideline_section(config=config, section_id=section_id)
print(json.dumps({
    "section_id": guideline_section["section_id"],
    "path": guideline_section["path"],
    "page_start": guideline_section["page_start"],
    "page_end": guideline_section["page_end"],
}, indent=2))
print()
print(guideline_section["text"][:4000])

{
  "section_id": "section-i-conventions-general-coding-guidelines-and-chapter-specific-guidelines-8-b-tabular-list-abbreviations-9",
  "path": [
    "Section I. Conventions, general coding guidelines and chapter specific guidelines .............................. 8",
    "b. Tabular List abbreviations ...................................................................................................... 9"
  ],
  "page_start": 2,
  "page_end": 2
}

7. Punctuation ............................................................................................................................... 9
8. Use of “and”. ........................................................................................................................... 10
9. Other and Unspecified codes .................................................................................................. 10


: 

## 6. Browse Tabular Chapters

Use this when you want the manual headings first, for example the chapter that contains the A and B code ranges before opening a narrower section.

In [ ]:
tabular_chapters = list_tabular_chapters(config=config, code_prefix="C", limit=20)
print(json.dumps(tabular_chapters, indent=2))

## 7. Open a Tabular Chapter and Section

Once you choose a chapter, inspect its section headings and then open one section to see the top-level codes directly beneath it.

In [ ]:
chapter_id = tabular_chapters["results"][0]["chapter_id"]
tabular_chapter = open_tabular_chapter(config=config, chapter_id=chapter_id)
print(json.dumps(tabular_chapter, indent=2)[:12000])

section_id = tabular_chapter["sections"][0]["section_id"]
tabular_section = open_tabular_section(config=config, section_id=section_id)
print()
print(json.dumps(tabular_section, indent=2)[:12000])